# Week 13B — Generative Vision and Synthetic Data

**MCTA 4364 Machine Vision** · Session 2 of Week 13 · 80 minutes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hasanzaki/MCTA-4364-Machine-Vision/blob/main/notebooks/W13_vision_language/W13B_generative_synthetic_data.ipynb) [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/hasanzaki/MCTA-4364-Machine-Vision/blob/main/notebooks/W13_vision_language/W13B_generative_synthetic_data.ipynb)

A factory makes thousands of good parts for every defective one, and some defects have never been seen at all. Collecting and labelling enough
examples is slow and expensive — so we increasingly **make** training data: render it, composite it, inject defects procedurally, or **generate**
it with diffusion models. Today you will train a small **diffusion model** from scratch on a laptop, learn how to **evaluate** generated images,
and run the experiment that matters in practice: do **synthetic defects** fix a classifier that has only five real examples of each defect?

### Learning outcomes
By the end of this session you will be able to:
1. Name the main sources of synthetic data (procedural rendering, domain randomisation, copy–paste, defect injection, generative models) and their trade-offs.
2. Explain the **diffusion** forward and reverse processes, the noise-prediction loss and **classifier-free guidance**.
3. Train a small class-conditional diffusion model and sample from it.
4. Evaluate generated data with the **Fréchet distance**, conditional accuracy and a **two-sample classifier test**, and always on **real** test data.
5. Use synthetic defects to rebalance a rare-defect problem and measure the gain.
6. Discuss licensing, provenance and misuse of generative models.

### Session plan (80 min)
| Time | Section | Activity |
|---|---|---|
| 0–10 | 1–2 | Why synthetic data; the families of generative models |
| 10–25 | 3 | Diffusion step by step: forward noising (interactive), a 2-D toy model |
| 25–45 | 4–5 | Train a diffusion model on part crops (runs while we discuss); evaluate it |
| 45–65 | 6–7 | Text-to-image models; synthetic defects for rare classes — the experiment |
| 65–72 | 8–9 | Sim-to-real, validation rules, responsible use |
| 72–80 | 12 | Quiz and exit ticket |
| Home | 10–11 | Exercises with self-checks, challenge |

**Hardware:** everything runs on a laptop CPU (the diffusion model trains in ~3–4 min on a CPU, under a minute on a 4 GB GPU). MobileCLIP (Week 13A)
is reused for evaluation. The optional Stable Diffusion demo in Section 6 needs a GPU or Colab.

## 0. Setup

In [ ]:
# Setup: works in Google Colab, Kaggle, Jupyter/VS Code on your laptop, and CI.
import importlib.util
import os
import subprocess
import sys

REPO_URL = "https://github.com/hasanzaki/MCTA-4364-Machine-Vision.git"
if not os.path.exists("resources/scripts/cvhelpers.py"):
    if os.path.exists("../../resources/scripts/cvhelpers.py"):  # opened from its own folder
        os.chdir("../..")
    else:  # Colab / Kaggle: fetch the course helpers
        if not os.path.isdir("MCTA-4364-Machine-Vision"):
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
        os.chdir("MCTA-4364-Machine-Vision")
print("Working directory:", os.getcwd())

# Install only what is missing. PyTorch is deliberately NOT pip-installed here:
# that would replace your laptop's CUDA build with a CPU-only one (see SETUP.md).
needed = {"cv2": "opencv-python", "skimage": "scikit-image", "ipywidgets": "ipywidgets",
          "matplotlib": "matplotlib", "scipy": "scipy", "plotly": "plotly",
          "open_clip": "open_clip_torch"}
missing = [pkg for mod, pkg in needed.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
if importlib.util.find_spec("torch") is None:
    raise ImportError("PyTorch is missing. Install it with CUDA support as described in "
                      "SETUP.md, or use Google Colab where it is preinstalled.")
sys.path.append("resources/scripts")

In [ ]:
import math
import time

import cv2
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import skimage.data
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy import linalg

import partsim
from cvhelpers import SMOKE, check, interact, pipeline_diagram, quiz
from dlhelpers import count_params, hw_profile, set_seed
from vlmhelpers import load_clip

plt.rcParams.update({"figure.dpi": 100, "axes.titlesize": 11})
HW = hw_profile()
DEV = HW["device"]
set_seed(0)
NAMES = partsim.SCENE_CLASSES


def rgb(img):
    return cv2.cvtColor(img, cv2.COLOR_GRAY2RGB) if img.ndim == 2 else img[..., ::-1]

## 1. Why synthetic data?

| problem with real data | how synthetic data helps |
|---|---|
| **rare classes** (defects are 0.1 % of production; new defect types have no examples) | generate or inject as many as needed |
| **labelling cost** (a mask can take minutes) | labels come for free and are pixel-perfect |
| **coverage** (night shift lighting, a new camera, a new product variant) | vary the conditions systematically |
| **privacy / safety** (people, accidents, medical images) | no real person or event is recorded |

The main sources, from cheapest to most flexible:
1. **Procedural rendering** — a program draws the scene from parameters (our `partsim`; professionally: Blender/BlenderProc, NVIDIA Omniverse Replicator, Unity Perception).
2. **Domain randomisation** (Tobin et al., 2017) — randomise everything that should *not* matter (lighting, colours, textures, noise) so that the real world looks like "just another variation".
3. **Copy–paste** (Ghiasi et al., 2021) — cut labelled objects out and paste them onto new backgrounds.
4. **Defect injection** — draw or paste defects onto images of good parts.
5. **Generative models** — learn the image distribution and sample new images (GANs, diffusion; text-to-image foundation models).

In [ ]:
# Helper: defect injection on images of good parts (explained and used in Section 7)
SP = partsim.SPEC
DEFECTS = ["missing_hole", "scratch", "chip"]
CLASSES4 = ["ok"] + DEFECTS


def part_geometry(info):
    cx, cy = info["centre"]
    th, px = np.deg2rad(info["angle_deg"]), 1 / info["mm_per_px"]
    holes = [(cx + SP["pcd"] / 2 * px * np.sin(th + np.pi / 4 + k * np.pi / 2), cy - SP["pcd"] / 2 * px * np.cos(th + np.pi / 4 + k * np.pi / 2))
             for k in range(4)]
    return (cx, cy), info["outer_d_mm"] / 2 * px, holes, th, px


def inject_defect(img, info, defect, rng):
    out = img.astype(np.float32)
    g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    (cx, cy), R, holes, th, px = part_geometry(info)
    metal = (g > np.percentile(g, 40)).astype(np.uint8)
    if defect == "scratch":
        for _ in range(30):                                            # retry until the scratch lies on metal
            r = rng.uniform(SP["centre_d"] / 2 + 2, SP["outer_d"] / 2 - 2.5) * px
            a = rng.uniform(0, 2 * np.pi)
            p = np.array([cx + r * np.cos(a), cy + r * np.sin(a)])
            d = rng.uniform(0, np.pi)
            L = rng.uniform(5, 11) * px
            u, n = np.array([np.cos(d), np.sin(d)]), np.array([-np.sin(d), np.cos(d)])
            pts = np.array([p + u * s * L / 2 + n * rng.normal(0, 0.8) * abs(s) for s in np.linspace(-1, 1, 5)])
            m = np.zeros(g.shape, np.float32)
            cv2.polylines(m, [np.round(pts * 4).astype(np.int32)], False, 1.0, int(rng.uniform(1.0, 1.8) * 4), cv2.LINE_AA, shift=2)
            if (m * metal).sum() >= 0.85 * m.sum():
                break
        out -= rng.uniform(40, 85) * cv2.GaussianBlur(m, (0, 0), 0.5)[..., None]
    elif defect == "chip":
        a = th + rng.uniform(0.6, 2 * np.pi - 0.6)                     # not at the orientation notch
        rim = np.array([cx + R * np.sin(a), cy - R * np.cos(a)])
        rc = rng.uniform(1.4, 2.4) * px
        ang = np.linspace(0, 2 * np.pi, 9, endpoint=False)
        pts = rim + rc * np.stack([np.cos(ang), np.sin(ang)], 1) * rng.uniform(0.6, 1.3, (9, 1))
        m = np.zeros(g.shape, np.uint8)
        cv2.fillPoly(m, [np.round(pts).astype(np.int32)], 1)
        m = cv2.GaussianBlur((m & metal).astype(np.float32), (0, 0), 0.7)[..., None]
        out = out * (1 - m) + (np.median(img[metal == 0], 0) + rng.normal(0, 3, img.shape)) * m
    elif defect == "missing_hole":
        hx, hy = holes[rng.integers(4)]
        rr = SP["bolt_d"] / 2 * px + 1.5
        yy, xx = np.mgrid[0:g.shape[0], 0:g.shape[1]]
        dist = np.hypot(xx - hx, yy - hy)
        m = np.clip(rr - dist + 0.5, 0, 1).astype(np.float32)[..., None]
        ring = (dist > rr + 1) & (dist < rr + 5) & (metal > 0)
        out = out * (1 - m) + (np.median(img[ring], 0) + rng.normal(0, 3, img.shape)) * m
    return np.clip(out, 0, 255).astype(np.uint8)

In [ ]:
rng = np.random.default_rng(3)
scene, sboxes, slabels, smasks = partsim.make_scene(7, n_objects=5, return_masks=True)


def randomise(img, rng):
    """Domain randomisation: colour gains, gamma, blur, noise and a tinted belt."""
    out = img.astype(np.float32) / 255
    out = out ** rng.uniform(0.6, 1.6) * rng.uniform(0.6, 1.3, 3)
    out = cv2.GaussianBlur(out, (0, 0), rng.uniform(0.1, 1.5))
    out = out + rng.normal(0, rng.uniform(0.005, 0.05), out.shape)
    return np.clip(out * 255, 0, 255).astype(np.uint8)


def copy_paste(img, masks, background):
    bg = cv2.resize(background, img.shape[1::-1])
    bg = cv2.cvtColor(bg, cv2.COLOR_GRAY2BGR) if bg.ndim == 2 else bg
    m = masks.any(0)[..., None]
    return np.where(m, img, bg)


ok_img, ok_info = partsim.make_part(5, "ok")
panels = [(scene, "1. procedural rendering (partsim)"),
          (np.hstack([randomise(scene, rng)[:, :320], randomise(scene, rng)[:, 320:]]), "2. domain randomisation (two variations)"),
          (copy_paste(scene, smasks, skimage.data.gravel()), "3. copy-paste onto a new background"),
          (inject_defect(ok_img, ok_info, "chip", np.random.default_rng(1)), "4. defect injection: a synthetic chip (Section 7)")]
fig, axes = plt.subplots(1, 4, figsize=(26, 5))
for ax, (im, title) in zip(axes, panels):
    ax.imshow(rgb(im)); ax.set_title(title); ax.axis("off")
fig.suptitle("Figure 1.1 - Four cheap ways to make labelled images (generative models follow)", fontweight="bold")
plt.tight_layout(); plt.show()

## 2. Generative models in one table

A generative model learns $p(\mathbf{x})$ — or $p(\mathbf{x}\mid c)$ for a class or a text prompt $c$ — from examples, then **samples** new images.

| family | idea | + | − | examples |
|---|---|---|---|---|
| **VAE** (2013) | encode to a latent Gaussian, decode back; maximise a lower bound on likelihood | stable, fast sampling, useful latent space | blurry images | the latent space of Stable Diffusion |
| **GAN** (2014) | a generator fools a discriminator | sharp, one forward pass | unstable training, mode collapse | StyleGAN, defect GANs |
| **Autoregressive** | predict pixels / image tokens one at a time | exact likelihood, scales like LLMs | slow sampling | PixelCNN, VQGAN-transformers |
| **Diffusion** (2020) | learn to remove noise step by step | best quality and diversity, stable training | many network evaluations per image | DDPM, Stable Diffusion, Imagen |
| **Flow matching** (2023) | learn a velocity field that moves noise to data along straight paths | fewer steps, simple loss | newer | Stable Diffusion 3, FLUX |

Diffusion and flow matching dominate today; we build a diffusion model because its maths is short and it trains on a laptop.

## 3. Diffusion step by step

**Forward process** (fixed, no learning): add Gaussian noise over $T$ steps with a schedule $\beta_t$. With $\alpha_t = 1-\beta_t$ and
$\bar\alpha_t = \prod_{s\le t}\alpha_s$, any step can be sampled in one go:

$$q(\mathbf{x}_t \mid \mathbf{x}_0) = \mathcal{N}\big(\sqrt{\bar\alpha_t}\,\mathbf{x}_0,\ (1-\bar\alpha_t)\mathbf{I}\big)
\quad\Longleftrightarrow\quad \mathbf{x}_t = \sqrt{\bar\alpha_t}\,\mathbf{x}_0 + \sqrt{1-\bar\alpha_t}\,\boldsymbol\epsilon,\ \ \boldsymbol\epsilon\sim\mathcal N(0,\mathbf I).$$

At $t = T$ the image is pure noise. **Training** (Ho et al., 2020): pick an image, a random $t$ and noise $\boldsymbol\epsilon$, and train a network
$\boldsymbol\epsilon_\theta(\mathbf{x}_t, t)$ to predict the noise:

$$\mathcal{L} = \mathbb{E}_{\mathbf{x}_0, t, \boldsymbol\epsilon}\ \big\lVert \boldsymbol\epsilon - \boldsymbol\epsilon_\theta(\mathbf{x}_t, t)\big\rVert^2 .$$

**Sampling** runs the chain backwards from noise, $t = T \dots 1$:

$$\mathbf{x}_{t-1} = \frac{1}{\sqrt{\alpha_t}}\Big(\mathbf{x}_t - \frac{1-\alpha_t}{\sqrt{1-\bar\alpha_t}}\,\boldsymbol\epsilon_\theta(\mathbf{x}_t,t)\Big) + \sqrt{\beta_t}\,\mathbf{z},
\qquad \mathbf{z}\sim\mathcal N(0,\mathbf I)\ (\mathbf{z}=0 \text{ at the last step}).$$

We use the **cosine schedule** (Nichol & Dhariwal, 2021), which destroys information more evenly than the original linear one.

In [ ]:
T = 200


def cosine_alpha_bar(T, s=0.008):
    t = np.linspace(0, T, T + 1)
    f = np.cos((t / T + s) / (1 + s) * np.pi / 2) ** 2
    return np.clip(f[1:] / f[0], 1e-5, 0.9999)


ALPHA_BAR = cosine_alpha_bar(T)
ALPHA = np.r_[ALPHA_BAR[0], ALPHA_BAR[1:] / ALPHA_BAR[:-1]]
BETA = 1 - ALPHA
LIN_BETA = np.linspace(1e-4, 0.02 * 1000 / T, T)
LIN_ALPHA_BAR = np.cumprod(1 - LIN_BETA)


def q_sample_ref(x0, t, alpha_bar, eps):
    return np.sqrt(alpha_bar[t]) * x0 + np.sqrt(1 - alpha_bar[t]) * eps


crops, crop_y = partsim.make_crops(30 if SMOKE else 300, seed=0, size=32)
X01 = crops.astype(np.float32) / 127.5 - 1                            # images scaled to [-1, 1]
washer = X01[list(crop_y).index(1)]
eps0 = np.random.default_rng(0).normal(size=washer.shape)
steps = [0, 10, 30, 60, 100, 150, 199]
fig = plt.figure(figsize=(24, 5.5))
for k, t in enumerate(steps):
    ax = fig.add_subplot(2, len(steps), k + 1)
    ax.imshow(q_sample_ref(washer, t, ALPHA_BAR, eps0), cmap="gray", vmin=-1.5, vmax=1.5); ax.axis("off")
    ax.set_title(f"t = {t}: signal {np.sqrt(ALPHA_BAR[t]):.2f}")
ax = fig.add_subplot(2, 1, 2)
ax.plot(ALPHA_BAR, color="#1a73e8", lw=2.5, label="cosine schedule (used here)")
ax.plot(LIN_ALPHA_BAR, color="#d93025", ls="--", label="linear schedule (DDPM 2020)")
ax.set_xlabel("step t"); ax.set_ylabel("alpha_bar_t (signal fraction)"); ax.legend(); ax.grid(alpha=0.3)
fig.suptitle("Figure 3.1 - The forward process: a washer dissolves into noise; the schedule sets how fast", fontweight="bold")
plt.tight_layout(); plt.show()

### 🎛️ Interactive: at which step can you no longer see the part?

In [ ]:
def noise_demo(t=60, part="washer"):
    x0 = X01[list(crop_y).index(NAMES.index(part))]
    xt = q_sample_ref(x0, t, ALPHA_BAR, eps0)
    snr = ALPHA_BAR[t] / (1 - ALPHA_BAR[t])
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    for ax, im, title in zip(axes, [x0, xt, xt - np.sqrt(ALPHA_BAR[t]) * x0], ["x0", f"x_t, t={t}", "the noise the network must predict (scaled)"]):
        ax.imshow(im, cmap="gray"); ax.set_title(title); ax.axis("off")
    plt.suptitle(f"signal-to-noise ratio {snr:.3g} ({10 * np.log10(snr):.1f} dB)"); plt.show()


interact(noise_demo, t=widgets.IntSlider(value=60, min=0, max=T - 1), part=widgets.Dropdown(options=NAMES, value="washer"))

### A diffusion model you can watch: 2-D points

Before images, the same algorithm on 2-D points: the "data" are points on a ring and a small cluster (think of them as two kinds of object).
A 3-layer MLP learns $\boldsymbol\epsilon_\theta(\mathbf{x}_t, t)$ in a few seconds; sampling moves random points back onto the data.

In [ ]:
def toy_data(n, rng):
    k = rng.random(n) < 0.7
    a = rng.uniform(0, 2 * np.pi, n)
    ring = np.c_[np.cos(a), np.sin(a)] * 1.5 + rng.normal(0, 0.05, (n, 2))
    blob = rng.normal(0, 0.15, (n, 2)) + [0.0, 0.0]
    return np.where(k[:, None], ring, blob).astype(np.float32)


class ToyDenoiser(nn.Module):
    def __init__(self, h=128):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(2 + 16, h), nn.SiLU(), nn.Linear(h, h), nn.SiLU(), nn.Linear(h, h), nn.SiLU(), nn.Linear(h, 2))

    def forward(self, x, t):
        f = torch.exp(-math.log(1000) * torch.arange(8) / 8)
        a = t[:, None].float() * f[None]
        return self.net(torch.cat([x, a.sin(), a.cos()], 1))


torch.manual_seed(0)
toy = ToyDenoiser()
opt = torch.optim.Adam(toy.parameters(), 2e-3)
ab_t = torch.tensor(ALPHA_BAR, dtype=torch.float32)
trng = np.random.default_rng(0)
t0 = time.perf_counter()
for it in range(300 if SMOKE else 4000):
    x0 = torch.from_numpy(toy_data(512, trng))
    t = torch.randint(0, T, (512,)); eps = torch.randn_like(x0); a = ab_t[t][:, None]
    loss = F.mse_loss(toy(a.sqrt() * x0 + (1 - a).sqrt() * eps, t), eps)
    opt.zero_grad(); loss.backward(); opt.step()
print(f"toy model trained in {time.perf_counter() - t0:.0f} s, final loss {loss.item():.3f}")

with torch.no_grad():
    x = torch.randn(1500, 2); snaps = {T: x.numpy().copy()}
    for t in reversed(range(T)):
        e = toy(x, torch.full((len(x),), t))
        x = (x - BETA[t] / math.sqrt(1 - ALPHA_BAR[t]) * e) / math.sqrt(ALPHA[t])
        if t > 0:
            x = x + math.sqrt(BETA[t]) * torch.randn_like(x)
        if t in (150, 100, 50, 0):
            snaps[t] = x.numpy().copy()
real_pts = toy_data(1500, np.random.default_rng(1))
fig, axes = plt.subplots(1, 6, figsize=(26, 4.6))
for ax, (t, pts) in zip(axes, sorted(snaps.items(), reverse=True)):
    ax.scatter(pts[:, 0], pts[:, 1], s=2, alpha=0.5, color="#1a73e8"); ax.set_title(f"reverse process, t = {t}")
axes[5].scatter(real_pts[:, 0], real_pts[:, 1], s=2, alpha=0.4, color="#9aa0a6", label="training data")
axes[5].scatter(snaps[0][:, 0], snaps[0][:, 1], s=2, alpha=0.4, color="#1a73e8", label="generated")
axes[5].set_title("generated vs training data"); axes[5].legend(loc="upper right", fontsize=8, markerscale=4)
for ax in axes:
    ax.set_xlim(-3, 3); ax.set_ylim(-3, 3); ax.set_aspect("equal"); ax.grid(alpha=0.2)
fig.suptitle("Figure 3.2 - Reverse diffusion on 2-D points: noise is gradually pulled onto the ring and the cluster", fontweight="bold")
plt.tight_layout(); plt.show()

## 4. Train a diffusion model on part crops

We now train $\boldsymbol\epsilon_\theta(\mathbf{x}_t, t, c)$ on 32 × 32 grey crops of the four conveyor parts, **conditioned** on the class $c$.
The network is a small **U-Net** (Ronneberger et al., 2015 — the same encoder–decoder with skip connections as Week 11A), with the time step and
the class added as embeddings inside every block.

**Classifier-free guidance** (Ho & Salimans, 2022): during training the class is replaced by a "no class" token 15 % of the time, so one network
learns both $\boldsymbol\epsilon_\theta(\mathbf{x}_t, t, c)$ and $\boldsymbol\epsilon_\theta(\mathbf{x}_t, t, \varnothing)$. At sampling time we
extrapolate away from the unconditional prediction with a guidance weight $w$:

$$\tilde{\boldsymbol\epsilon} = (1 + w)\,\boldsymbol\epsilon_\theta(\mathbf{x}_t, t, c) - w\,\boldsymbol\epsilon_\theta(\mathbf{x}_t, t, \varnothing).$$

Larger $w$ → more typical, class-faithful images, less diversity. Text-to-image models use exactly this with a text prompt as $c$.

In [ ]:
class Block(nn.Module):
    def __init__(self, cin, cout, emb):
        super().__init__()
        self.c1, self.c2 = nn.Conv2d(cin, cout, 3, padding=1), nn.Conv2d(cout, cout, 3, padding=1)
        self.n1, self.n2 = nn.GroupNorm(4, cout), nn.GroupNorm(4, cout)
        self.emb = nn.Linear(emb, cout)
        self.skip = nn.Conv2d(cin, cout, 1) if cin != cout else nn.Identity()

    def forward(self, x, e):
        h = F.silu(self.n1(self.c1(x))) + self.emb(e)[:, :, None, None]       # add time + class information
        return F.silu(self.n2(self.c2(h))) + self.skip(x)


class TinyUNet(nn.Module):
    def __init__(self, n_classes=4, ch=(16, 32, 64), emb=64):
        super().__init__()
        self.emb_dim = emb
        self.cls = nn.Embedding(n_classes + 1, emb)                          # the last index is "no class"
        self.time = nn.Sequential(nn.Linear(emb, emb), nn.SiLU(), nn.Linear(emb, emb))
        self.inp = nn.Conv2d(1, ch[0], 3, padding=1)
        self.d1, self.d2, self.d3 = Block(ch[0], ch[0], emb), Block(ch[0], ch[1], emb), Block(ch[1], ch[2], emb)
        self.mid = Block(ch[2], ch[2], emb)
        self.u3, self.u2, self.u1 = Block(2 * ch[2], ch[1], emb), Block(2 * ch[1], ch[0], emb), Block(2 * ch[0], ch[0], emb)
        self.out = nn.Conv2d(ch[0], 1, 3, padding=1)

    def forward(self, x, t, c):
        half = self.emb_dim // 2
        f = torch.exp(-math.log(10000) * torch.arange(half, device=x.device) / half)
        a = t[:, None].float() * f[None]
        e = self.time(torch.cat([a.sin(), a.cos()], 1)) + self.cls(c)
        h1 = self.d1(self.inp(x), e)                                   # 32 x 32
        h2 = self.d2(F.avg_pool2d(h1, 2), e)                           # 16 x 16
        h3 = self.d3(F.avg_pool2d(h2, 2), e)                           # 8 x 8
        m = self.mid(F.avg_pool2d(h3, 2), e)                           # 4 x 4
        u = self.u3(torch.cat([F.interpolate(m, scale_factor=2), h3], 1), e)
        u = self.u2(torch.cat([F.interpolate(u, scale_factor=2), h2], 1), e)
        u = self.u1(torch.cat([F.interpolate(u, scale_factor=2), h1], 1), e)
        return self.out(u)


NULL = len(NAMES)
if SMOKE:
    ITERS, BATCH, CH = 30, 32, (16, 32, 64)
elif HW["cuda"]:
    ITERS, BATCH, CH = 6000, 128, (32, 64, 128)
else:
    ITERS, BATCH, CH = 2000, 64, (16, 32, 64)
torch.manual_seed(0)
unet = TinyUNet(ch=CH).to(DEV)
print(f"TinyUNet: {count_params(unet) / 1e6:.2f} M parameters | {ITERS} iterations, batch {BATCH} on {DEV}")
Xt = torch.from_numpy(X01)[:, None].to(DEV)
yt = torch.from_numpy(crop_y).to(DEV)
abar = torch.tensor(ALPHA_BAR, dtype=torch.float32, device=DEV)
opt = torch.optim.AdamW(unet.parameters(), 2e-3)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, 2e-3, total_steps=ITERS)
losses = []
t0 = time.perf_counter()
for it in range(ITERS):
    idx = torch.randint(0, len(Xt), (BATCH,), device=DEV)
    x0, c = Xt[idx], yt[idx].clone()
    c[torch.rand(BATCH, device=DEV) < 0.15] = NULL                         # classifier-free guidance dropout
    x0 = torch.where(torch.rand(BATCH, 1, 1, 1, device=DEV) < 0.5, x0.flip(3), x0)
    t = torch.randint(0, T, (BATCH,), device=DEV)
    eps = torch.randn_like(x0)
    a = abar[t][:, None, None, None]
    loss = F.mse_loss(unet(a.sqrt() * x0 + (1 - a).sqrt() * eps, t, c), eps)
    opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    losses.append(loss.item())
train_time = time.perf_counter() - t0
print(f"trained in {train_time / 60:.1f} min, final loss {np.mean(losses[-50:]):.4f}")

In [ ]:
def ddpm_step_ref(x_t, eps_cond, eps_uncond, t, w, alpha, alpha_bar, beta, z):
    eps = (1 + w) * eps_cond - w * eps_uncond
    x = (x_t - (1 - alpha[t]) / np.sqrt(1 - alpha_bar[t]) * eps) / np.sqrt(alpha[t])
    return x + np.sqrt(beta[t]) * z if t > 0 else x


@torch.no_grad()
def sample(classes, w=2.0, seed=0, keep=()):
    """Reverse diffusion for a list of class indices; returns images in [-1, 1] and snapshots at the steps in `keep`."""
    unet.eval()
    g = torch.Generator(device="cpu").manual_seed(seed)
    c = torch.as_tensor(classes, device=DEV)
    x = torch.randn(len(c), 1, 32, 32, generator=g).to(DEV)
    snaps = {}
    for t in reversed(range(T)):
        tt = torch.full((len(c),), t, device=DEV)
        e_c, e_u = unet(x, tt, c), unet(x, tt, torch.full_like(c, NULL))
        z = torch.randn(x.shape, generator=g).to(DEV)
        x = ddpm_step_ref(x, e_c, e_u, t, w, ALPHA, ALPHA_BAR, BETA, z)
        if t in keep:
            snaps[t] = x.cpu().numpy()[:, 0]
    unet.train()
    return x.clamp(-1, 1).cpu().numpy()[:, 0], snaps


to_u8 = lambda a: ((np.clip(a, -1, 1) + 1) * 127.5).astype(np.uint8)
N_SHOW = 8
t0 = time.perf_counter()
gen, snaps = sample(np.repeat(np.arange(4), N_SHOW), w=2.0, keep=(150, 100, 60, 30, 10))
print(f"{len(gen)} samples x {T} steps in {time.perf_counter() - t0:.1f} s")
fig, axes = plt.subplots(1, 3, figsize=(24, 6.5), gridspec_kw=dict(width_ratios=[1, 1, 0.9]))
grid = lambda ims: np.vstack([np.hstack(list(ims[r * N_SHOW:(r + 1) * N_SHOW])) for r in range(4)])
axes[0].imshow(grid(np.concatenate([crops[crop_y == c][:N_SHOW] for c in range(4)])), cmap="gray"); axes[0].set_title("real crops")
axes[1].imshow(grid(to_u8(gen)), cmap="gray"); axes[1].set_title("generated (guidance w = 2)")
for ax in axes[:2]:
    ax.set_yticks(np.arange(4) * 32 + 16); ax.set_yticklabels(NAMES); ax.set_xticks([])
axes[2].plot(np.convolve(losses, np.ones(50) / 50, mode="valid"), color="#1a73e8"); axes[2].set_yscale("log")
axes[2].set_xlabel("iteration"); axes[2].set_ylabel("noise-prediction MSE (moving average)"); axes[2].grid(alpha=0.3); axes[2].set_title("training loss")
fig.suptitle(f"Figure 4.1 - A {count_params(unet) / 1e6:.2f} M-parameter diffusion model after {train_time / 60:.1f} min of training", fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(22, 7.5))
seq = [np.hstack([np.clip(snaps[t][k * N_SHOW], -1.5, 1.5) for t in (150, 100, 60, 30, 10)] + [gen[k * N_SHOW]]) for k in range(4)]
axes[0].imshow(np.hstack([np.pad(s, 2, constant_values=1.5) for s in seq]), cmap="gray"); axes[0].axis("off")
axes[0].set_title("reverse process for one sample per class: t = 150, 100, 60, 30, 10, 0 (left to right in each group)")
ws = [0.0, 1.0, 3.0, 6.0]
rows_ = [np.hstack(list(to_u8(sample([NAMES.index("flange")] * 10, w=w, seed=1)[0]))) for w in ws]
axes[1].imshow(np.vstack(rows_), cmap="gray"); axes[1].set_xticks([])
axes[1].set_yticks(np.arange(len(ws)) * 32 + 16); axes[1].set_yticklabels([f"w = {w:g}" for w in ws])
axes[1].set_title("guidance weight for the class 'flange': higher w -> more typical, less varied (same random seeds)")
fig.suptitle("Figure 4.2 - Denoising in action, and classifier-free guidance", fontweight="bold")
plt.tight_layout(); plt.show()

**What to notice.** Round, repetitive shapes (washers, flanges, nuts) are learned quickly; **thin, oriented** structures (bolts) are the hardest
and still look melted after a few minutes of training — more data, more iterations or a larger network fix that. Without guidance ($w = 0$) the
samples are more diverse but less clearly one class.

## 5. How good are generated images?

"They look fine" is not a metric. Three standard checks:

1. **Fréchet distance** (FID, Heusel et al., 2017): embed real and generated images with a pretrained network, fit a Gaussian to each set, and
   compare them:
   $$\mathrm{FD} = \lVert \boldsymbol\mu_r - \boldsymbol\mu_g \rVert^2 + \operatorname{tr}\big(\Sigma_r + \Sigma_g - 2(\Sigma_r\Sigma_g)^{1/2}\big).$$
   Lower is better; it measures both quality and diversity. FID uses Inception-v3 features; newer work uses CLIP or DINOv2 features. Absolute values
   depend on the embedding and the **number of samples** (FD is biased upward for small sets), so only compare numbers computed the same way — we
   include a **real-vs-real** baseline.
2. **Conditional accuracy**: does a classifier trained on real images recognise the class we asked for?
3. **Train on synthetic, test on real (TSTR)** — the only check that answers the practical question; we do it in Section 7.

In [ ]:
clip_model = load_clip("mobileclip")


def frechet_ref(A, B):
    mu1, mu2 = A.mean(0), B.mean(0)
    s1, s2 = np.cov(A, rowvar=False), np.cov(B, rowvar=False)
    covmean = linalg.sqrtm(s1 @ s2)
    return float(np.sum((mu1 - mu2) ** 2) + np.trace(s1 + s2 - 2 * np.real(covmean)))


def linear_probe(X, y, n_classes, weight_decay=1e-3, class_weight=None):
    Xt_, yt_ = torch.tensor(X), torch.tensor(y)
    W = torch.zeros(X.shape[1], n_classes, requires_grad=True)
    b = torch.zeros(n_classes, requires_grad=True)
    cw = None if class_weight is None else torch.tensor(class_weight, dtype=torch.float32)
    opt_ = torch.optim.LBFGS([W, b], max_iter=300)

    def closure():
        opt_.zero_grad()
        loss_ = F.cross_entropy(30 * Xt_ @ W + b, yt_, weight=cw) + weight_decay * (W ** 2).sum()
        loss_.backward()
        return loss_

    opt_.step(closure)
    return lambda Z: (30 * torch.tensor(Z) @ W + b).argmax(1).numpy()


N_EVAL = 16 if SMOKE else 60
t0 = time.perf_counter()
gen_eval, _ = sample(np.repeat(np.arange(4), N_EVAL), w=2.0, seed=7)
gen_w0, _ = sample(np.repeat(np.arange(4), N_EVAL), w=0.0, seed=7)
hold, hold_y = partsim.make_crops(N_EVAL, seed=99, size=32)
up = lambda ims: [cv2.resize(im, (128, 128), interpolation=cv2.INTER_CUBIC) for im in ims]
ref = np.arange(len(crops))[:4 * N_EVAL]                            # a real reference set of the same size as the others
E_real = clip_model.encode_images(up(crops[ref]))
E_hold = clip_model.encode_images(up(hold))
E_gen = clip_model.encode_images(up(to_u8(gen_eval)))
E_gen0 = clip_model.encode_images(up(to_u8(gen_w0)))
E_blur = clip_model.encode_images(up([cv2.GaussianBlur(im, (0, 0), 1.5) for im in hold]))
print(f"sampled and embedded in {time.perf_counter() - t0:.0f} s")
fds = {"real vs real (held out)": frechet_ref(E_real, E_hold), "generated, w = 2": frechet_ref(E_real, E_gen),
       "generated, w = 0": frechet_ref(E_real, E_gen0), "real, blurred": frechet_ref(E_real, E_blur)}
probe = linear_probe(E_real, crop_y[ref], 4)
gen_y = np.repeat(np.arange(4), N_EVAL)
cond_acc = [np.mean(probe(E_gen)[gen_y == c] == c) for c in range(4)]
real_acc = [np.mean(probe(E_hold)[hold_y == c] == c) for c in range(4)]
fig, axes = plt.subplots(1, 2, figsize=(18, 4.8))
axes[0].barh(list(fds)[::-1], list(fds.values())[::-1], color=["#9aa0a6", "#f29900", "#1a73e8", "#34a853"])
axes[0].set_xlabel("Fréchet distance to the real training crops (MobileCLIP features, lower is better)"); axes[0].grid(alpha=0.3, axis="x")
axes[0].set_title("distribution distance")
xs = np.arange(4)
axes[1].bar(xs - 0.2, real_acc, 0.4, label="held-out REAL crops", color="#34a853")
axes[1].bar(xs + 0.2, cond_acc, 0.4, label="GENERATED crops (w = 2)", color="#1a73e8")
axes[1].set_xticks(xs); axes[1].set_xticklabels(NAMES); axes[1].set_ylim(0, 1.1); axes[1].legend(loc="lower left"); axes[1].grid(alpha=0.3, axis="y")
axes[1].set_title("conditional accuracy: a probe trained on real crops, applied to each set")
fig.suptitle("Figure 5.1 - Measuring the generator: every sample is recognisable as its class, yet the set is still far from the real distribution", fontweight="bold")
plt.tight_layout(); plt.show()
for k, v in fds.items():
    print(f"FD {k:<26} {v:.3f}")

**Read the two panels together.** The conditional accuracy says every generated sample *looks like* its class to a classifier; the Fréchet
distance says the *set* of samples is still roughly ten times further from the real distribution than a second real set — details, textures and
variety are missing. Guidance makes samples more typical but less varied, which is why $w = 0$ can have the lower FD. A blurred real set scores
worst: FD is very sensitive to lost detail.

## 6. Text-to-image foundation models

Our model knows four classes at 32 × 32 pixels. **Latent diffusion** (Rombach et al., 2022 — Stable Diffusion) scales the same idea: a VAE
compresses 512 × 512 images to a 64 × 64 latent, a large U-Net or transformer denoises the latent, and a **text encoder (CLIP!)** provides the
condition with classifier-free guidance. Distilled versions (SD-Turbo, SDXL-Turbo, FLUX-schnell) need only 1–4 steps.

Tools for **synthetic training data** built on them:
| tool | what it does | use for |
|---|---|---|
| **inpainting** | regenerate a masked region from a prompt | put a defect / object exactly where the label says |
| **ControlNet** (Zhang et al., 2023) | condition on edges, depth, a segmentation map | keep the geometry of a real part, change its appearance |
| **image-to-image** | start the reverse process from a noised real image | many variations of one real defect |
| **DreamBooth / LoRA** fine-tuning | teach the model your object from 5–20 photos | generate *your* part, not a generic one |

The cell below runs **SD-Turbo** (one step) if a GPU and Hugging Face are available. Look critically at the results: pretty, but are they
*correct* parts with *labelled* defects?

> **Licences.** SD-Turbo: Stability AI non-commercial research licence. Check every model licence before using its images to train a commercial
> product, and record which model and prompt produced each synthetic image (provenance).

In [ ]:
if SMOKE or not HW["cuda"]:
    print("Stable Diffusion demo skipped (needs a GPU; ~2.5 GB download). Run it in Colab with a T4 GPU.")
else:
    try:
        from diffusers import AutoPipelineForText2Image
        pipe = AutoPipelineForText2Image.from_pretrained("stabilityai/sd-turbo", torch_dtype=torch.float16, variant="fp16").to("cuda")
        pipe.enable_attention_slicing()
        prompts = ["a close-up photo of a machined steel flange with four bolt holes on a black conveyor belt, industrial lighting",
                   "a close-up photo of a steel washer with a deep scratch, macro, industrial inspection",
                   "a photo of hex nuts and bolts scattered on a dark conveyor belt, top view",
                   "a photo of a cracked metal gear, factory, harsh lighting"]
        ims = [pipe(prompt=p, num_inference_steps=1, guidance_scale=0.0, generator=torch.Generator("cuda").manual_seed(k)).images[0]
               for k, p in enumerate(prompts)]
        fig, axes = plt.subplots(1, 4, figsize=(24, 6.5))
        for ax, im, p in zip(axes, ims, prompts):
            ax.imshow(im); ax.axis("off"); ax.set_title(p[:60] + "...", fontsize=9)
        fig.suptitle("Figure 6.1 - SD-Turbo, one denoising step per image", fontweight="bold")
        plt.tight_layout(); plt.show()
        del pipe; torch.cuda.empty_cache()
    except Exception as exc:
        print(f"Stable Diffusion demo skipped: {type(exc).__name__}: {exc}")

## 7. Synthetic defects for rare classes — the experiment

Back to the inspection station of Week 7A. The line produced **200 good parts** but only **5 examples each** of *missing hole*, *scratch* and
*chip*. A classifier trained on this is biased towards "ok" — the costliest possible mistake.

We **inject** synthetic defects into images of good parts, with the simple drawing code of `inject_defect` (defined after the setup) (deliberately *not* the code that made the real
defects — in reality you never have the true generator):
- **scratch** — a thin, slightly curved dark polyline on the metal;
- **chip** — an irregular bite out of the rim, filled with belt;
- **missing hole** — one bolt hole filled with the surrounding metal colour (we know where the holes are from the part geometry, as a pose estimate
  would tell us — Week 7A).

Then we compare four training sets on the **same real, balanced test set** (50 real parts per class), using MobileCLIP embeddings and a linear
probe (Week 13A) so that each run takes seconds.

In [ ]:
N_OK, N_REAL, N_SYN, N_TEST = (20, 3, 10, 8) if SMOKE else (200, 5, 100, 50)
t0 = time.perf_counter()
ok_imgs = [partsim.make_part(10000 + k, "ok")[0] for k in range(N_OK)]
real_def = {d: [partsim.make_part(20000 + 100 * i + k, d)[0] for k in range(N_REAL)] for i, d in enumerate(DEFECTS)}
irng = np.random.default_rng(0)
donors = [partsim.make_part(30000 + k, "ok") for k in range(N_SYN)]
syn_def = {d: [inject_defect(im, inf, d, irng) for im, inf in donors] for d in DEFECTS}
test_imgs, test_y = [], []
for i, d in enumerate(CLASSES4):
    for k in range(N_TEST):
        test_imgs.append(partsim.make_part(90000 + 1000 * i + k, d)[0]); test_y.append(i)
test_y = np.array(test_y)
fig, axes = plt.subplots(3, 8, figsize=(24, 9.6))
for r, d in enumerate(DEFECTS):
    for k in range(5):
        axes[r, k].imshow(rgb(syn_def[d][k])); axes[r, k].set_title(f"SYNTHETIC {d}", fontsize=9, color="#1a73e8")
    for k in range(3):
        axes[r, 5 + k].imshow(rgb(test_imgs[list(test_y).index(r + 1) + k])); axes[r, 5 + k].set_title(f"REAL {d}", fontsize=9, color="#d93025")
for ax in axes.ravel():
    ax.axis("off")
fig.suptitle("Figure 7.1 - Injected defects (left, on images of good parts) vs real defects (right): similar, not identical", fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
emb = lambda ims: clip_model.encode_images(ims)
E_ok = emb(ok_imgs)
E_rd = {d: emb(v) for d, v in real_def.items()}
E_sd = {d: emb(v) for d, v in syn_def.items()}
E_te = emb(test_imgs)
print(f"embedded {N_OK + 3 * N_REAL + 3 * N_SYN + 4 * N_TEST} images in {time.perf_counter() - t0:.0f} s")
R_ = np.concatenate([E_rd[d] for d in DEFECTS]); yR = np.repeat([1, 2, 3], N_REAL)
S_ = np.concatenate([E_sd[d] for d in DEFECTS]); yS = np.repeat([1, 2, 3], N_SYN)
yO = np.zeros(N_OK, int)
CONDITIONS = {
    "real only (200 ok : 5 per defect)": (np.r_[E_ok, R_], np.r_[yO, yR], False),
    "real only, class-weighted loss": (np.r_[E_ok, R_], np.r_[yO, yR], True),
    "real + synthetic defects": (np.r_[E_ok, R_, S_], np.r_[yO, yR, yS], True),
    "synthetic defects only (no real defect)": (np.r_[E_ok, S_], np.r_[yO, yS], True),
}
recalls = {}
for name, (X, y, weighted) in CONDITIONS.items():
    cw = len(y) / (4 * np.bincount(y, minlength=4)) if weighted else None
    pred = linear_probe(X, y, 4, class_weight=cw)(E_te)
    recalls[name] = np.array([np.mean(pred[test_y == c] == c) for c in range(4)])
fig, ax = plt.subplots(figsize=(18, 5.2))
xs = np.arange(5)
cols = ["#9aa0a6", "#f29900", "#1a73e8", "#a142f4"]
for k, (name, r) in enumerate(recalls.items()):
    vals = np.r_[r, r.mean()]
    ax.bar(xs + (k - 1.5) * 0.2, vals, 0.2, color=cols[k], label=name)
    ax.text(4 + (k - 1.5) * 0.2, vals[-1] + 0.02, f"{vals[-1]:.2f}", ha="center", fontsize=9)
ax.set_xticks(xs); ax.set_xticklabels([f"recall: {c}" for c in CLASSES4] + ["BALANCED ACCURACY"]); ax.set_ylim(0, 1.12)
ax.legend(loc="upper center", ncol=4, fontsize=9); ax.grid(alpha=0.3, axis="y")
ax.set_title(f"Figure 7.2 - Train on synthetic, test on REAL: {N_TEST} real parts per class", fontweight="bold")
plt.show()
for name, r in recalls.items():
    print(f"{name:<42} " + " ".join(f"{c} {v:.2f}" for c, v in zip(CLASSES4, r)) + f" | balanced {r.mean():.3f}")

**Can a classifier tell our synthetic defects from real ones?** A **two-sample classifier test** (C2ST, Lopez-Paz & Oquab, 2017): train a
classifier to separate real from synthetic examples of the same class and measure its cross-validated accuracy. 50 % = indistinguishable (in this
feature space); 100 % = a clear **sim-to-real gap** that a model could exploit.

In [ ]:
def c2st(A, B, folds=5, seed=0):
    X = np.r_[A, B]; y = np.r_[np.zeros(len(A), int), np.ones(len(B), int)]
    idx = np.random.default_rng(seed).permutation(len(y))
    accs = []
    for f in range(folds):
        te = idx[f::folds]; tr = np.setdiff1d(idx, te)
        accs.append(np.mean(linear_probe(X[tr], y[tr], 2)(X[te]) == y[te]))
    return float(np.mean(accs))


gap = {d: c2st(E_te[test_y == i + 1], E_sd[d][:N_TEST]) for i, d in enumerate(DEFECTS)}
gap["ok vs ok (control)"] = c2st(E_te[test_y == 0], E_ok[:N_TEST])
fig, ax = plt.subplots(figsize=(10, 3.8))
ax.barh(list(gap), list(gap.values()), color=["#1a73e8"] * 3 + ["#9aa0a6"]); ax.axvline(0.5, color="k", ls=":")
ax.set_xlim(0, 1); ax.set_xlabel("real-vs-synthetic classifier accuracy (0.5 = indistinguishable)"); ax.grid(alpha=0.3, axis="x")
ax.set_title("Figure 7.3 - Two-sample classifier test per defect class", fontweight="bold")
plt.show()

**Interpretation.** Five real examples per class are not enough, and re-weighting the loss helps only partly. Adding a hundred injected defects
per class lifts the recall of every defect class, even though a classifier can still tell many synthetic images from real ones: the synthetic
defects do not have to be perfect, they have to **cover the variation** the real ones show (position, size, orientation). Training on synthetic
defects **only** works here because our injector happens to be close to the real process — on a real line, always keep some real examples and
always **test on real data only**.

## 8. Sim-to-real: rules that keep you honest

1. **Test on real data only.** Synthetic validation scores are meaningless for deployment (TSTR, Section 7).
2. **Randomise what should not matter**, keep what must matter realistic (defect *size* for a size-based reject rule; do not randomise it away).
3. **Mix, do not replace**: a small fraction of real images anchors the model (typical: 10–50 % real).
4. **Measure the gap** (FD, C2ST) and look at the **failure cases** on real data — they tell you what the generator is missing.
5. **Beware leakage**: a generative model fine-tuned on your test images will copy them. Keep the test set out of *every* stage, including generator training.
6. **Label consistency**: synthetic labels are perfect; real labels are not. A mismatch in how boxes are drawn (tight vs loose) costs mAP (Week 9A).

In [ ]:
fig, ax = plt.subplots(figsize=(22, 2.6))
pipeline_diagram(["few real\nimages", "generator / renderer /\ninjector", "synthetic set\n(+ free labels)", "mix with\nreal", "train",
                  "TEST ON REAL\n(TSTR)", "inspect failures ->\nimprove generator"],
                 colors=["#f1f3f4", "#fef7e0", "#e8f0fe", "#e8f0fe", "#e6f4ea", "#fce8e6", "#fef7e0"], ax=ax, fontsize=10)
ax.set_title("Figure 8.1 - The synthetic-data loop: the real test set is the referee", fontweight="bold")
plt.show()

## 9. Responsible use of generative models

| issue | what to do |
|---|---|
| **licences** | model weights and their outputs have terms (non-commercial, attribution, AGPL); check before training a product on them |
| **copyright of training data** | large text-to-image models were trained on scraped images; some jurisdictions and customers restrict their use |
| **provenance** | keep a record (model, version, prompt, seed) for every synthetic image; C2PA content credentials and watermarks (e.g. SynthID) mark generated media |
| **disclosure** | the EU AI Act requires that AI-generated or manipulated images (deepfakes) are labelled as such |
| **misuse** | fake defect photos in a quality report, fake evidence, impersonation — generated images must never be passed off as real records |
| **bias and gaps** | a generator reproduces the gaps of its training data; it will not invent the defect nobody has seen |

## 10. Exercises (with self-checks)
Replace each `raise NotImplementedError`, then run the check below it.

### E1. Forward diffusion in one step\nReturn $\mathbf{x}_t = \sqrt{\bar\alpha_t}\,\mathbf{x}_0 + \sqrt{1-\bar\alpha_t}\,\boldsymbol\epsilon$ (Section 3). `t` is an integer index into `alpha_bar`.

In [ ]:
def q_sample(x0, t, alpha_bar, eps):
    # TODO: your code here
    raise NotImplementedError

In [ ]:
def _test_e1():
    x0 = np.ones((4, 4)); eps = np.full((4, 4), 2.0); ab = np.array([0.99, 0.64, 0.25])
    assert np.allclose(q_sample(x0, 1, ab, eps), 0.8 + 0.6 * 2), "x_t = sqrt(ab) x0 + sqrt(1 - ab) eps"
    assert np.allclose(q_sample(x0, 2, ab, eps), 0.5 + np.sqrt(0.75) * 2)
    r = np.random.default_rng(0); x = r.normal(size=(64, 64)); e = r.normal(size=(64, 64))
    assert np.allclose(q_sample(x, 150, ALPHA_BAR, e), q_sample_ref(x, 150, ALPHA_BAR, e))
    big = q_sample(np.zeros(20000), T - 1, ALPHA_BAR, r.normal(size=20000))
    assert abs(big.std() - 1) < 0.03, "at t = T the result should be (almost) pure unit noise"


check("E1 q_sample", _test_e1)

### E2. Fréchet distance\nGiven two sets of embeddings `A` ($n\times d$) and `B` ($m\times d$), return the Fréchet distance of Section 5. Use `np.cov(..., rowvar=False)` and `scipy.linalg.sqrtm`, and keep only the real part of the matrix square root.

In [ ]:
def frechet_distance(A, B):
    # TODO: your code here
    raise NotImplementedError

In [ ]:
def _test_e2():
    r = np.random.default_rng(0)
    A = r.normal(size=(4000, 3))
    assert frechet_distance(A, A) < 1e-6, "a set has distance 0 to itself"
    d = frechet_distance(A, A + np.array([3.0, 0.0, 0.0]))
    assert abs(d - 9.0) < 1e-6, f"a pure shift by 3 gives 9, got {d:.4f}"
    B = r.normal(size=(4000, 3)) * 2.0
    assert abs(frechet_distance(A, B) - 3.0) < 0.3, "N(0, I) vs N(0, 4I) in 3-D: 3 (1 + 4 - 2*2) = 3"
    assert abs(frechet_distance(E_real, E_gen) - fds["generated, w = 2"]) < 1e-6


check("E2 frechet_distance", _test_e2)

### E3. One guided reverse step
Implement one reverse step with classifier-free guidance (Sections 3 and 4): combine the two noise predictions with weight `w`, compute
$\mathbf{x}_{t-1}$, and add $\sqrt{\beta_t}\,\mathbf{z}$ **except at `t == 0`**. It must work for NumPy arrays and PyTorch tensors (use only
arithmetic on the inputs and `math.sqrt`/`np.sqrt` on the scalar schedule values).

In [ ]:
def ddpm_step(x_t, eps_cond, eps_uncond, t, w, alpha, alpha_bar, beta, z):
    # TODO: your code here
    raise NotImplementedError

In [ ]:
def _test_e3():
    a, ab, b = np.array([0.9, 0.8]), np.array([0.9, 0.72]), np.array([0.1, 0.2])
    x, ec, eu, z = np.full(3, 1.0), np.full(3, 0.5), np.full(3, 0.1), np.full(3, 1.0)
    got = ddpm_step(x, ec, eu, 1, 0.0, a, ab, b, z)
    want = (1 - 0.2 / np.sqrt(0.28) * 0.5) / np.sqrt(0.8) + np.sqrt(0.2)
    assert np.allclose(got, want), "check the update formula (w = 0)"
    got = ddpm_step(x, ec, eu, 1, 2.0, a, ab, b, z)
    want = (1 - 0.2 / np.sqrt(0.28) * (3 * 0.5 - 2 * 0.1)) / np.sqrt(0.8) + np.sqrt(0.2)
    assert np.allclose(got, want), "guidance: eps = (1 + w) eps_cond - w eps_uncond"
    assert np.allclose(ddpm_step(x, ec, eu, 0, 0.0, a, ab, b, z), (1 - 0.1 / np.sqrt(0.1) * 0.5) / np.sqrt(0.9)), "no noise at t = 0"
    xt = torch.randn(2, 1, 4, 4)
    out = ddpm_step(xt, xt * 0.1, xt * 0.2, 50, 2.0, ALPHA, ALPHA_BAR, BETA, torch.zeros_like(xt))
    assert torch.is_tensor(out) and torch.allclose(out, ddpm_step_ref(xt, xt * 0.1, xt * 0.2, 50, 2.0, ALPHA, ALPHA_BAR, BETA, torch.zeros_like(xt)))


check("E3 ddpm_step", _test_e3)

### E4. A better defect injector (no self-check)
Look at Figure 7.1 and the C2ST of Figure 7.3. Improve `inject_defect` for the class with the largest real-vs-synthetic gap (for example: scratch
width and darkness, chip shape and size, lighting-consistent shading). Re-run Section 7. Did the C2ST accuracy drop? Did the real recall change?
Is a lower C2ST always better for the downstream task?

In [ ]:
# E4: your code here

## 11. Challenge: a stain detector with zero real stains
`partsim` also makes a sixth class, **stain** (`partsim.make_part(seed, "stain")`), which none of today's models has seen. Write an injector for
stains using only good parts, train the Section 7 classifier with five classes (ok + four defects), and evaluate on real stains. Then train the
diffusion model of Section 4 on 64 × 64 defect crops instead of whole parts (GPU recommended) and compare its samples with your injector using FD and C2ST.

In [ ]:
# Your challenge code here

## 12. Quick check (auto-marked)

In [ ]:
quiz([
    {
        "q": 'What does domain randomisation vary?',
        "options": [
            'The labels',
            'Nuisance factors such as lighting, colours, textures and noise',
            'The test set',
            'The number of classes',
        ],
        "answer": '4e1994852c',
        "explain": 'So that the real world looks like one more variation.',
    },
    {
        "q": 'In a diffusion model, what does the network predict during training?',
        "options": [
            'The class',
            'The noise that was added to the image',
            'The next pixel',
            'The image resolution',
        ],
        "answer": 'b5ab554e45',
        "explain": 'Loss = || eps - eps_theta(x_t, t) ||^2.',
    },
    {
        "q": 'Why can x_t be sampled directly from x0 without running t steps?',
        "options": [
            'It cannot',
            'Sums of Gaussian noise are Gaussian, giving q(x_t | x0) in closed form',
            'Because T is small',
            'Because of the U-Net',
        ],
        "answer": '1d3418c63c',
        "explain": 'x_t = sqrt(alpha_bar) x0 + sqrt(1 - alpha_bar) eps.',
    },
    {
        "q": 'What does a larger classifier-free guidance weight w do?',
        "options": [
            'More diverse samples',
            'More typical, class-faithful but less diverse samples',
            'Faster training',
            'Bigger images',
        ],
        "answer": 'd105641a48',
        "explain": 'It extrapolates away from the unconditional prediction.',
    },
    {
        "q": 'Why is a real-vs-real baseline reported next to FD values?',
        "options": [
            'It is always zero',
            'FD depends on the features and the sample size, so absolute values need a reference',
            'To make the numbers smaller',
            'It replaces the test set',
        ],
        "answer": 'c918c2c48a',
        "explain": 'Only compare FDs computed the same way.',
    },
    {
        "q": 'A model trained with synthetic defects scores 99% on a synthetic validation set. What do you do next?',
        "options": [
            'Deploy it',
            'Evaluate it on real defect images',
            'Generate more synthetic data',
            'Lower the learning rate',
        ],
        "answer": '8a78ad062a',
        "explain": 'Train on synthetic, test on real.',
    },
    {
        "q": 'A two-sample classifier separates real from synthetic defects with 95% accuracy. What does it show?',
        "options": [
            'The synthetic data are useless',
            'There is a clear sim-to-real gap in that feature space',
            'The real data are wrong',
            'Nothing',
        ],
        "answer": 'f487e126ba',
        "explain": 'Synthetic data can still help (Section 7), but the gap is worth closing.',
    },
    {
        "q": 'Which statement about generated images is correct?',
        "options": [
            'They are always free to use',
            'Their use can be restricted by model licences and must be traceable',
            'They never need labelling as AI-generated',
            'They cannot contain bias',
        ],
        "answer": '60bf43de1c',
        "explain": 'Record provenance; follow licences and disclosure rules.',
    },
], title='Week 13B quiz')

## 13. Exit ticket (reflection)
1. Your line has 3 photos of a new crack defect. List two ways to make 500 training examples and how you would check they are useful.
2. Why did synthetic defects help even though a classifier could often tell them from real ones?
3. One thing about diffusion models that surprised you today.

## 14. What's new (2025–26)

> - **Image generation**: flow-matching transformers (Stable Diffusion 3.5, FLUX.1) and fast distilled models (1–4 steps) make on-premise generation practical;
>   natively multimodal LLMs now generate and **edit** images from instructions.
> - **Controllable synthesis for vision datasets**: layout-, mask- and depth-conditioned generation (ControlNet successors, InstanceDiffusion-style
>   models) produce images *with* labels; video world models (e.g. NVIDIA Cosmos) generate driving and robotics training data.
> - **Industrial practice**: defect synthesis with inpainting/LoRA on a handful of real defects, NVIDIA Omniverse Replicator and BlenderProc
>   for rendered data, and the rule that synthetic data **supplement** — not replace — a real test set.
> - **Provenance**: C2PA content credentials and invisible watermarks are being built into cameras, generators and platforms.

## 15. References and further exploration
- Ho, J., Jain, A. & Abbeel, P. (2020). *Denoising diffusion probabilistic models.* NeurIPS. [arXiv:2006.11239](https://arxiv.org/abs/2006.11239)
- Nichol, A. & Dhariwal, P. (2021). *Improved denoising diffusion probabilistic models.* ICML. [arXiv:2102.09672](https://arxiv.org/abs/2102.09672)
- Ho, J. & Salimans, T. (2022). *Classifier-free diffusion guidance.* [arXiv:2207.12598](https://arxiv.org/abs/2207.12598)
- Rombach, R. et al. (2022). *High-resolution image synthesis with latent diffusion models.* CVPR. [arXiv:2112.10752](https://arxiv.org/abs/2112.10752)
- Lipman, Y. et al. (2023). *Flow matching for generative modeling.* ICLR. [arXiv:2210.02747](https://arxiv.org/abs/2210.02747)
- Goodfellow, I. et al. (2014). *Generative adversarial nets.* NeurIPS. [arXiv:1406.2661](https://arxiv.org/abs/1406.2661)
- Kingma, D. P. & Welling, M. (2014). *Auto-encoding variational Bayes.* ICLR. [arXiv:1312.6114](https://arxiv.org/abs/1312.6114)
- Heusel, M. et al. (2017). *GANs trained by a two time-scale update rule converge to a local Nash equilibrium* (FID). NeurIPS. [arXiv:1706.08500](https://arxiv.org/abs/1706.08500)
- Lopez-Paz, D. & Oquab, M. (2017). *Revisiting classifier two-sample tests.* ICLR. [arXiv:1610.06545](https://arxiv.org/abs/1610.06545)
- Tobin, J. et al. (2017). *Domain randomization for transferring deep neural networks from simulation to the real world.* IROS. [arXiv:1703.06907](https://arxiv.org/abs/1703.06907)
- Ghiasi, G. et al. (2021). *Simple copy-paste is a strong data augmentation method for instance segmentation.* CVPR. [arXiv:2012.07177](https://arxiv.org/abs/2012.07177)
- Zhang, L., Rao, A. & Agrawala, M. (2023). *Adding conditional control to text-to-image diffusion models* (ControlNet). ICCV. [arXiv:2302.05543](https://arxiv.org/abs/2302.05543)
- Ronneberger, O., Fischer, P. & Brox, T. (2015). *U-Net.* MICCAI. [arXiv:1505.04597](https://arxiv.org/abs/1505.04597)
- Denninger, M. et al. (2023). *BlenderProc2: A procedural pipeline for photorealistic rendering.* JOSS 8(82). [doi:10.21105/joss.04901](https://doi.org/10.21105/joss.04901)

**Data:** all images are procedural (`partsim.py`) or from scikit-image (`gravel`); SD-Turbo outputs are generated at run time only.

## 16. Key takeaways
- Synthetic data attack **rarity, labelling cost and coverage**: rendering, domain randomisation, copy–paste, defect injection and generative models.
- **Diffusion**: fixed forward noising, a network trained to predict the noise, iterative denoising; **classifier-free guidance** trades diversity for fidelity.
- Evaluate generators with **FD** (with a real-vs-real baseline), **conditional accuracy** and a **two-sample test** — and decide with **train-on-synthetic, test-on-real**.
- A few hundred imperfect synthetic defects can rescue a classifier that has seen only five real ones; the real test set stays the referee.
- Generated images carry **licence, provenance and disclosure** obligations.

**Next week (W14):** project demonstrations — bring your pipeline, your data story and your honest evaluation.